In [30]:
import pickle
import os
import numpy as np
import json
import kagglehub
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from torch.utils.data import DataLoader
import torch.optim as optim
import time

DATA_DIR = os.path.join("..", "data")

In [2]:
with open(os.path.join(DATA_DIR, "als_model.pkl"), "rb") as f:
    model = pickle.load(f)
with open(os.path.join(DATA_DIR, "anilist_to_mal.json"), "r") as f:
    mal_ids = json.load(f)
with open(os.path.join(DATA_DIR, "anime_data.jsonl"), "r") as f:
    anime_content = [json.loads(line) for line in f]

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test = pd.read_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))


In [3]:
anime_tag_vectors = np.load(os.path.join(DATA_DIR, "anime_tag_vectors.npy"))
manga_tag_vectors = np.load(os.path.join(DATA_DIR, "manga_tag_vectors.npy"))

als_user_factors = model.user_factors 
als_item_factors = model.item_factors 

In [4]:
anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()}

# AniList ID -> real MAL ID (via idMal crosswalk; drop failed lookups)
anilist_to_mal = {int(k): v for k, v in mal_ids.items() if v is not None}

none_count = sum(1 for v in mal_ids.values() if v is None)
print(f"anilist_to_mal: {none_count}/{len(mal_ids)} AniList entries had no MAL match (idMal was null) -- dropped")

# real MAL ID -> ratings dataset's internal animeID (bridge step that was missing before)
animes['mal_id'] = animes['mal_url'].str.extract(r'/anime/(\d+)').astype(int)

# --- sanity-check the bridge tables before trusting a dict built from them ---
dup_mal = animes['mal_id'].duplicated(keep=False)
dup_animeid = animes['animeID'].duplicated(keep=False)
if dup_mal.any():
    print(f"WARNING: {dup_mal.sum()} rows in animes share a duplicated mal_id -- "
          f"dict(zip(...)) will silently keep only the last row per key")
if dup_animeid.any():
    print(f"WARNING: {dup_animeid.sum()} rows in animes share a duplicated animeID")
if not dup_mal.any() and not dup_animeid.any():
    print("animes['mal_id'] and animes['animeID'] are both unique")

mal_to_animeid = dict(zip(animes['mal_id'], animes['animeID']))

# --- walk the full chain: AniList idx -> real MAL id -> dataset animeID -> ALS row ---
aligned_rows = []
for i, a in enumerate(anime_content):
    anilist_id = a['id']
    real_mal_id = anilist_to_mal.get(anilist_id)
    if real_mal_id is None:
        continue  # no MAL match for this AniList entry


    animeid = mal_to_animeid.get(real_mal_id)
    if animeid is None:
        continue  # MAL id doesn't appear in the ratings dataset at all

    als_row = anime_id_map_reverse.get(animeid)
    if als_row is None:
        continue  # in the ratings dataset, but never rated in `train` -> no ALS row

    aligned_rows.append({
        'anilist_idx': i,
        'real_mal_id': real_mal_id,
        'animeid': animeid,
        'als_row': als_row,
    })

aligned_df = pd.DataFrame(aligned_rows)
print(f"Aligned {len(aligned_df)} / {len(anime_content)} AniList anime through all three ID systems to an ALS row")


anilist_to_mal: 11/4950 AniList entries had no MAL match (idMal was null) -- dropped
animes['mal_id'] and animes['animeID'] are both unique
Aligned 4778 / 5000 AniList anime through all three ID systems to an ALS row


In [5]:
def anilist_title(a):
    # AniList title is a dict, not a plain string
    t = a['title']
    return t.get('english') or t.get('romaji') or t.get('native')

animes_title_by_id = animes.set_index('animeID')['title']

sample = aligned_df.sample(min(10, len(aligned_df)), random_state=2)
for _, row in sample.iterrows():
    anilist_t = anilist_title(anime_content[row['anilist_idx']])
    ratings_t = animes_title_by_id.loc[row['animeid']]
    print(f"AniList: {anilist_t!r:55} | animes: {ratings_t!r}")


AniList: 'Legend of the Galactic Heroes Gaiden: A Hundred Billion Stars' | animes: 'Legend of the Galactic Heroes Gaiden'
AniList: 'One Piece Special: Protect! The Last Great Performance' | animes: 'One Piece: Protect! The Last Great Performance'
AniList: 'Higehiro: After Being Rejected, I Shaved and Took in a High School Runaway' | animes: 'Higehiro: After Being Rejected, I Shaved and Took in a High School Runaway'
AniList: 'That Time I Got Reincarnated as a Slime Season 3'      | animes: 'That Time I Got Reincarnated as a Slime Season 3'
AniList: 'Detective Conan: The Scarlet Bullet'                   | animes: 'Detective Conan Movie 24: The Scarlet Bullet'
AniList: 'Cross Ange: Rondo of Angel and Dragon'                 | animes: 'Cross Ange: Rondo of Angel and Dragon'
AniList: 'Durarara!! Specials'                                   | animes: 'Durarara!! Specials'
AniList: 'Kaze no Stigma'                                        | animes: 'Kaze no Stigma'
AniList: 'BASTARD!! -Heavy M

In [6]:
for n in [64, 128, 200, 300]:
    svd_test = TruncatedSVD(n_components=n, random_state=42)
    svd_test.fit(anime_tag_vectors)
    print(f"{n} dims -> {svd_test.explained_variance_ratio_.sum():.2%} variance explained")

64 dims -> 44.10% variance explained
128 dims -> 65.04% variance explained
200 dims -> 81.43% variance explained
300 dims -> 95.03% variance explained


In [7]:
combined_item_features = []
for _, row in aligned_df.iterrows():
    als_vec = als_item_factors[row['als_row']]           
    tag_vec = anime_tag_vectors[row['anilist_idx']]   
    combined = np.concatenate([als_vec, tag_vec])     
    combined_item_features.append(combined)

combined_item_features = np.array(combined_item_features)

aligned_animeids = set(aligned_df['animeid'])

train_aligned = train[train['anime_id'].isin(aligned_animeids)]

print(f"Original train rows: {len(train):,}")
print(f"Filtered to aligned anime: {len(train_aligned):,}")
print(f"Unique users remaining: {train_aligned['user_id'].nunique():,}")

Original train rows: 118,387,327
Filtered to aligned anime: 112,708,907
Unique users remaining: 1,773,734


In [8]:
positive_counts = train_aligned[train_aligned['is_positive'] == 1].groupby('user_id').size()
qualifying_users = positive_counts[positive_counts >= 5].index
train_final = train_aligned[train_aligned['user_id'].isin(qualifying_users)]

print(f"Final training rows: {len(train_final):,}")
print(f"Final unique users: {train_final['user_id'].nunique():,}")

Final training rows: 110,749,861
Final unique users: 1,461,421


In [12]:
MAX_POSITIVES_PER_USER = 50

train_positive = train_final[train_final['is_positive'] == 1]
train_positive_shuffled = train_positive.sample(frac=1, random_state=42)
train_capped = train_positive_shuffled.groupby('user_id').head(MAX_POSITIVES_PER_USER)

print(f"Training pairs after capping: {len(train_capped):,}")

Training pairs after capping: 39,179,955


In [14]:
animeid_to_aligned_idx = {row['animeid']: i for i, row in aligned_df.reset_index(drop=True).iterrows()}
user_id_to_useridx = train_final[['user_id', 'user_idx']].drop_duplicates().set_index('user_id')['user_idx'].to_dict()

print(len(animeid_to_aligned_idx), len(user_id_to_useridx))

4730 1461421


In [9]:
import random
import torch
from torch.utils.data import Dataset

class TwoTowerDataset(Dataset):
    def __init__(self, train_capped, animeid_to_aligned_idx, user_id_to_useridx, aligned_animeids):
        # Only keep rows where the anime is actually in our aligned set
        # (should already be true given train_final's filtering, but worth being defensive)
        self.data = train_capped.reset_index(drop=True)
        self.animeid_to_aligned_idx = animeid_to_aligned_idx
        self.user_id_to_useridx = user_id_to_useridx
        self.aligned_animeids = list(aligned_animeids)  # for fast random.choice sampling

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        user_idx = self.user_id_to_useridx[row['user_id']]
        pos_anime_idx = self.animeid_to_aligned_idx[row['anime_id']]

        # Sample a negative anime at random from the aligned set
        neg_anime_id = random.choice(self.aligned_animeids)
        neg_anime_idx = self.animeid_to_aligned_idx[neg_anime_id]

        return {
            'user_idx': user_idx,
            'pos_item_idx': pos_anime_idx,
            'neg_item_idx': neg_anime_idx,
        }

In [15]:
aligned_animeids = set(aligned_df['animeid'])

dataset = TwoTowerDataset(train_capped, animeid_to_aligned_idx, user_id_to_useridx, aligned_animeids)

BATCH_SIZE = 8192*2

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# Pull one batch to confirm the shapes look right
batch = next(iter(dataloader))
print(batch['user_idx'].shape, batch['pos_item_idx'].shape, batch['neg_item_idx'].shape)

torch.Size([16384]) torch.Size([16384]) torch.Size([16384])


In [16]:
import torch.nn as nn

class UserTower(nn.Module):
    def __init__(self, input_dim=64, output_dim=64, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)


class ItemTower(nn.Module):
    def __init__(self, input_dim=486, output_dim=64, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [17]:
user_tower = UserTower()
item_tower = ItemTower()

optimizer = optim.Adam(
    list(user_tower.parameters()) + list(item_tower.parameters()),
    lr=0.001
)

def bpr_loss(user_vec, pos_vec, neg_vec):
    pos_score = (user_vec * pos_vec).sum(dim=1)
    neg_score = (user_vec * neg_vec).sum(dim=1)
    # sigmoid(pos - neg) close to 1 means the model correctly ranks pos above neg
    return -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()

In [41]:
CHECKPOINT_PATH = os.path.join(DATA_DIR, "two_tower_checkpoint.pt")
# --- At the very top, before training: try to resume if a checkpoint exists ---
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    user_tower = UserTower()
    item_tower = ItemTower()
    user_tower.load_state_dict(checkpoint['user_tower_state'])
    item_tower.load_state_dict(checkpoint['item_tower_state'])
    
    optimizer = optim.Adam(
        list(user_tower.parameters()) + list(item_tower.parameters()),
        lr=0.001
    )
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming from epoch {start_epoch + 1}")
else:
    user_tower = UserTower()
    item_tower = ItemTower()
    optimizer = optim.Adam(
        list(user_tower.parameters()) + list(item_tower.parameters()),
        lr=0.001
    )
    start_epoch = 0
    print("Starting fresh — no checkpoint found")

# --- Train ---
EPOCHS = 3  # the real total target, not just 1

for epoch in range(start_epoch, EPOCHS):
    total_loss = 0
    num_batches = 0
    
    for batch in dataloader:
        user_vecs = torch.tensor(als_user_factors[batch['user_idx']], dtype=torch.float32)
        pos_vecs = torch.tensor(combined_item_features[batch['pos_item_idx']], dtype=torch.float32)
        neg_vecs = torch.tensor(combined_item_features[batch['neg_item_idx']], dtype=torch.float32)
        
        user_out = user_tower(user_vecs)
        pos_out = item_tower(pos_vecs)
        neg_out = item_tower(neg_vecs)
        
        loss = bpr_loss(user_out, pos_out, neg_out)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if num_batches % 500 == 0:
            print(f"Epoch {epoch+1}, batch {num_batches}, avg loss: {total_loss/num_batches:.4f}")
    
    epoch_loss = total_loss / num_batches
    print(f"Epoch {epoch+1} complete — avg loss: {epoch_loss:.4f}")
    
    # --- Save after EVERY epoch ---
    torch.save({
        'epoch': epoch,
        'user_tower_state': user_tower.state_dict(),
        'item_tower_state': item_tower.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': epoch_loss,
    }, CHECKPOINT_PATH)
    print(f"Checkpoint saved after epoch {epoch+1}")

Starting fresh — no checkpoint found
Epoch 1, batch 500, avg loss: 0.2043
Epoch 1, batch 1000, avg loss: 0.1739
Epoch 1, batch 1500, avg loss: 0.1603
Epoch 1, batch 2000, avg loss: 0.1520
Epoch 1 complete — avg loss: 0.1475
Checkpoint saved after epoch 1
Epoch 2, batch 500, avg loss: 0.1224
Epoch 2, batch 1000, avg loss: 0.1213
Epoch 2, batch 1500, avg loss: 0.1205
Epoch 2, batch 2000, avg loss: 0.1198
Epoch 2 complete — avg loss: 0.1194
Checkpoint saved after epoch 2
Epoch 3, batch 500, avg loss: 0.1159
Epoch 3, batch 1000, avg loss: 0.1154
Epoch 3, batch 1500, avg loss: 0.1150
Epoch 3, batch 2000, avg loss: 0.1147
Epoch 3 complete — avg loss: 0.1144
Checkpoint saved after epoch 3


In [18]:
user_tower.eval()  # switches off dropout — you want consistent, deterministic output now, not training-time randomness
item_tower.eval()

with torch.no_grad():  # no need to track gradients anymore, saves memory/time
    all_item_vecs = torch.tensor(combined_item_features, dtype=torch.float32)
    all_item_embeddings = item_tower(all_item_vecs).numpy()

print(all_item_embeddings.shape)  # should be (4778, 64)

(4778, 64)


In [20]:
def two_tower_recommend_for_user(user_id, k=10):
    if user_id not in user_id_to_useridx:
        return []
    
    user_idx = user_id_to_useridx[user_id]
    user_vec = torch.tensor(als_user_factors[user_idx], dtype=torch.float32).unsqueeze(0)
    
    with torch.no_grad():
        user_embedding = user_tower(user_vec).numpy()[0]
    
    scores = all_item_embeddings @ user_embedding  # dot product against every aligned anime
    
    # Exclude anime this user already positively rated
    already_rated = set(train_final[
        (train_final['user_id'] == user_id) & (train_final['is_positive'] == 1)
    ]['anime_id'])
    
    aligned_animeid_list = aligned_df['animeid'].tolist()
    ranked_indices = scores.argsort()[::-1]
    
    recommendations = []
    for idx in ranked_indices:
        animeid = aligned_animeid_list[idx]
        if animeid not in already_rated:
            recommendations.append(animeid)
        if len(recommendations) == k:
            break
    
    return recommendations

In [26]:
anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories)) 

from scipy.sparse import load_npz

user_item_matrix_full = load_npz(os.path.join(DATA_DIR, "item_user_matrix.npz"))
user_item_matrix_full = user_item_matrix_full.T.tocsr()

print(user_item_matrix_full.shape)

def als_recommend_for_user(user_idx, k=10):
    recommended = model.recommend(
        user_idx,
        user_item_matrix_full[user_idx],
        N=k
    )
    item_indices, scores = recommended
    return [anime_id_map[i] for i in item_indices]

(1774060, 19903)


In [22]:
print([f for f in os.listdir(DATA_DIR) if f.endswith('.npz')])

['item_user_matrix.npz', 'item_similarity.npz']


In [27]:
test_user = list(user_id_to_useridx.keys())[0]

two_tower_recs = two_tower_recommend_for_user(test_user, k=10)
print("Two-Tower recs:", two_tower_recs)
user_id_to_useridx_verified = train[['user_id', 'user_idx']].drop_duplicates().set_index('user_id')['user_idx'].to_dict()

als_recs = als_recommend_for_user(user_id_to_useridx_verified[test_user], k=10)
print("ALS recs:", als_recs)
print("Overlap:", set(two_tower_recs) & set(als_recs))

Two-Tower recs: [2393, 8165, 1433, 629, 3061, 3494, 1061, 5778, 5235, 6335]
ALS recs: [481, 2708, 38, 772, 1825, 2897, 973, 1967, 3053, 1145]
Overlap: set()


In [ ]:
aligned_animeid_list = aligned_df['animeid'].tolist()
user_positive_lookup = train_final[train_final['is_positive'] == 1].groupby('user_id')['anime_id'].apply(set).to_dict()

# Step 1: compute ALL evaluated users' embeddings in ONE batched forward pass
def precision_recall_two_tower(test_df, k=10, sample_users=2000):
    aligned_animeids = set(aligned_df['animeid'])
    test_aligned = test_df[test_df['anime_id'].isin(aligned_animeids) & (test_df['is_positive'] == 1)]
    
    grouped = test_aligned.groupby('user_id')
    users_to_eval = list(grouped.groups.keys())
    users_to_eval = [u for u in users_to_eval if u in user_id_to_useridx]
    
    if sample_users and len(users_to_eval) > sample_users:
        users_to_eval = list(np.random.choice(users_to_eval, size=sample_users, replace=False))
    
    print(f"Evaluating {len(users_to_eval)} users...")
    
    # Batch compute every user's embedding at once, instead of one at a time
    user_indices = [user_id_to_useridx[u] for u in users_to_eval]
    user_vecs = torch.tensor(als_user_factors[user_indices], dtype=torch.float32)
    
    with torch.no_grad():
        user_embeddings = user_tower(user_vecs).numpy()  # shape: (num_users, 64)
    
    # Now score ALL users against ALL items in one matrix multiply
    all_scores = user_embeddings @ all_item_embeddings.T  # shape: (num_users, num_aligned_anime)
    
    total_relevant = 0
    total_hits = 0
    start = time.time()
    
    for i, user_id in enumerate(users_to_eval):
        actual_positive = set(grouped.get_group(user_id)['anime_id'])
        already_rated = user_positive_lookup.get(user_id, set())
        
        scores = all_scores[i]
        ranked_indices = scores.argsort()[::-1]
        
        recommended = []
        for idx in ranked_indices:
            animeid = aligned_animeid_list[idx]
            if animeid not in already_rated:
                recommended.append(animeid)
            if len(recommended) == k:
                break
        
        hits = len(actual_positive & set(recommended))
        total_hits += hits
        total_relevant += len(actual_positive)
        
        if (i + 1) % 200 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed
            print(f"{i+1}/{len(users_to_eval)} | {rate:.1f} users/sec")
    
    recall = total_hits / total_relevant if total_relevant > 0 else 0
    precision = total_hits / (len(users_to_eval) * k)
    
    return precision, recall

In [34]:
precision, recall = precision_recall_two_tower(test, k=10, sample_users=2000)
print(f"Two-Tower — Precision@10: {precision:.4f}")
print(f"Two-Tower — Recall@10: {recall:.4f}")

KeyboardInterrupt: 

In [33]:
# Step 1: compute ALL evaluated users' embeddings in ONE batched forward pass
def precision_recall_two_tower(test_df, k=10, sample_users=2000):
    aligned_animeids = set(aligned_df['animeid'])
    test_aligned = test_df[test_df['anime_id'].isin(aligned_animeids) & (test_df['is_positive'] == 1)]
    
    grouped = test_aligned.groupby('user_id')
    users_to_eval = list(grouped.groups.keys())
    users_to_eval = [u for u in users_to_eval if u in user_id_to_useridx]
    
    if sample_users and len(users_to_eval) > sample_users:
        users_to_eval = list(np.random.choice(users_to_eval, size=sample_users, replace=False))
    
    print(f"Evaluating {len(users_to_eval)} users...")
    
    # Batch compute every user's embedding at once, instead of one at a time
    user_indices = [user_id_to_useridx[u] for u in users_to_eval]
    user_vecs = torch.tensor(als_user_factors[user_indices], dtype=torch.float32)
    
    with torch.no_grad():
        user_embeddings = user_tower(user_vecs).numpy()  # shape: (num_users, 64)
    
    # Now score ALL users against ALL items in one matrix multiply
    all_scores = user_embeddings @ all_item_embeddings.T  # shape: (num_users, num_aligned_anime)
    
    total_relevant = 0
    total_hits = 0
    start = time.time()
    
    for i, user_id in enumerate(users_to_eval):
        actual_positive = set(grouped.get_group(user_id)['anime_id'])
        already_rated = user_positive_lookup.get(user_id, set())
        
        scores = all_scores[i]
        ranked_indices = scores.argsort()[::-1]
        
        recommended = []
        for idx in ranked_indices:
            animeid = aligned_animeid_list[idx]
            if animeid not in already_rated:
                recommended.append(animeid)
            if len(recommended) == k:
                break
        
        hits = len(actual_positive & set(recommended))
        total_hits += hits
        total_relevant += len(actual_positive)
        
        if (i + 1) % 200 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed
            print(f"{i+1}/{len(users_to_eval)} | {rate:.1f} users/sec")
    
    recall = total_hits / total_relevant if total_relevant > 0 else 0
    precision = total_hits / (len(users_to_eval) * k)
    
    return precision, recall